In [ ]:
# No depth dimension: removed
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten
from tensorflow.keras.regularizers import l2
from spektral.layers import GCSConv
from spektral.utils import normalized_adjacency

# Load training dataset
X_train = np.load('/path/to/training/data/X_train.npy')  # Replace with actual path
Y_train = np.load('/path/to/training/data/Y_train.npy')  # Replace with actual path

# Ensure correct input shape: (batch_size, num_nodes, num_features)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], -1))  # Merge last 2 dimensions

# Define a graph adjacency matrix (fully connected for now)
num_nodes = X_train.shape[1]  # Assuming 26 nodes per sample
A = np.ones((num_nodes, num_nodes))  # Fully connected graph
A = normalized_adjacency(A)  # Normalize adjacency matrix

# SGCN Model for Regression
def build_sgcn_model(input_shape, A_matrix, l2_reg=1e-4, dropout_rate=0.4):
    X_input = Input(shape=input_shape)
    A_input = Input(shape=(num_nodes, num_nodes))  # Graph adjacency matrix

    x = GCSConv(32, activation="relu", kernel_regularizer=l2(l2_reg))([X_input, A_input])
    x = Dropout(dropout_rate)(x)
    x = GCSConv(64, activation="relu", kernel_regularizer=l2(l2_reg))([x, A_input])
    x = Dropout(dropout_rate)(x)
    x = Flatten()(x)
    x = Dense(128, activation="relu", kernel_regularizer=l2(l2_reg))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation="linear")(x)  # Regression output

    model = Model(inputs=[X_input, A_input], outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='mse',
                  metrics=['mae'])
    return model

# Build and summarize the model
model = build_sgcn_model(X_train.shape[1:], A)
model.summary()

# Train model on training dataset only
history = model.fit([X_train, np.tile(A, (X_train.shape[0], 1, 1))], Y_train, batch_size=16, epochs=50, verbose=1)

# Save the model
model.save("sgcn_regression_model.h5")


In [ ]:
from spektral.layers import GCSConv
import tensorflow as tf

tf.keras.utils.get_custom_objects()["GCSConv"] = GCSConv


In [ ]:
model = tf.keras.models.load_model(
    "sgcn_regression_model.h5",
    custom_objects={
        'GCSConv': GCSConv,
        'mse': tf.keras.losses.MeanSquaredError()  # Fix missing 'mse'
    }
)


In [ ]:
model.compile(loss=tf.keras.losses.MeanSquaredError(), optimizer="adam")


In [ ]:
model.save("sgcn_regression_model_fixed.h5")


In [ ]:
from spektral.layers import GCSConv
import tensorflow as tf

# Register custom layers and loss function
tf.keras.utils.get_custom_objects()["GCSConv"] = GCSConv
tf.keras.utils.get_custom_objects()["mse"] = tf.keras.losses.MeanSquaredError()

# Load the trained model with correct custom objects
model = tf.keras.models.load_model(
    "sgcn_regression_model.h5",
    custom_objects={'GCSConv': GCSConv, 'mse': tf.keras.losses.MeanSquaredError()}
)


In [ ]:
import numpy as np
import tensorflow as tf
from spektral.utils import normalized_adjacency

# Load the trained model
model = tf.keras.models.load_model("sgcn_regression_model.h5", custom_objects={'GCSConv': tf.keras.layers.Layer})

# Load validation dataset
X_val = np.load('/path/to/validation/data/X_val.npy')  # Replace with actual path
Y_val = np.load('/path/to/validation/data/Y_val.npy')  # Replace with actual path

# Ensure X_val has the correct shape (batch_size, num_nodes, num_features)
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], -1))  # Merge last 2 dimensions

# Define the adjacency matrix (same as training)
num_nodes = X_val.shape[1]
A = np.ones((num_nodes, num_nodes))  # Fully connected graph
A = normalized_adjacency(A)  # Normalize adjacency matrix
A_batch = np.tile(A, (X_val.shape[0], 1, 1))  # Tile for batch processing

# Predict Y values using the model
Y_pred = model.predict([X_val, A_batch])

# Compute squared error cost function
squared_error = np.square(Y_pred.flatten() - Y_val)  # Element-wise squared difference
total_squared_error = np.sum(squared_error)  # Sum over all samples

# Print results
print(f"Squared Error Cost Function: {total_squared_error:.4f}")


In [ ]:
import numpy as np
import tensorflow as tf
from spektral.layers import GCSConv  # Import GCSConv
from spektral.utils import normalized_adjacency

# Load the trained model with GCSConv registered
model = tf.keras.models.load_model(
    "sgcn_regression_model.h5",
    custom_objects={'GCSConv': GCSConv}
)

# Load validation dataset
X_val = np.load('/path/to/training/data/X_train.npy')  # Replace with actual path
Y_val = np.load('/path/to/training/data/Y_train.npy')  # Replace with actual path

# Ensure X_val has the correct shape (batch_size, num_nodes, num_features)
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], -1))

# Define adjacency matrix (same as during training)
num_nodes = X_val.shape[1]
A = np.ones((num_nodes, num_nodes))  # Fully connected graph
A = normalized_adjacency(A)  # Normalize adjacency matrix
A_batch = np.tile(A, (X_val.shape[0], 1, 1))  # Tile for batch processing

# Predict Y values using the model
Y_pred = model.predict([X_val, A_batch])

# Compute squared error cost function
squared_error = np.square(Y_pred.flatten() - Y_val)  # Element-wise squared difference
total_squared_error = np.sum(squared_error)  # Sum over all samples

# Print results
print(f"Squared Error Cost Function: {total_squared_error:.4f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error
from spektral.layers import GCSConv  # Import GCSConv
from spektral.utils import normalized_adjacency

# Load the trained model with GCSConv registered
model = tf.keras.models.load_model(
    "sgcn_regression_model.h5",
    custom_objects={'GCSConv': GCSConv}
)

# Load test dataset
X_test = np.load('/path/to/testing/data/X_test.npy')
Y_test = np.load('/path/to/testing/data/Y_test.npy')

# Ensure X_test has the correct shape (batch_size, num_nodes, num_features)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], -1))

# Define adjacency matrix (fully connected graph as used in validation)
num_nodes = X_test.shape[1]
A = np.ones((num_nodes, num_nodes))  # Fully connected graph
A = normalized_adjacency(A)  # Normalize adjacency matrix
A_batch = np.tile(A, (X_test.shape[0], 1, 1))  # Tile for batch processing

# Predict Y values using the model
Y_pred = model.predict([X_test, A_batch])

# Compute RMSE
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))

# Print RMSE result
print(f"Root Mean Squared Error (RMSE) on Test Set: {rmse:.4f}")


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_squared_error
from spektral.layers import GCSConv  # Import GCSConv
from spektral.utils import normalized_adjacency
from sklearn.metrics import r2_score

# Load the trained model with GCSConv registered
model = tf.keras.models.load_model(
    "sgcn_regression_model.h5",
    custom_objects={'GCSConv': GCSConv}
)

# Load test dataset
X_test = np.load('/path/to/testing/data/X_test.npy')
Y_test = np.load('/path/to/testing/data/Y_test.npy')

# Ensure X_test has the correct shape (batch_size, num_nodes, num_features)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], -1))

# Define adjacency matrix (fully connected graph as used in validation)
num_nodes = X_test.shape[1]
A = np.ones((num_nodes, num_nodes))  # Fully connected graph
A = normalized_adjacency(A)  # Normalize adjacency matrix
A_batch = np.tile(A, (X_test.shape[0], 1, 1))  # Tile for batch processing

# Predict Y values using the model
Y_pred = model.predict([X_test, A_batch])

# Compute R² Score
r2 = r2_score(Y_test, Y_pred)

print(f"Coefficient of Determination (R²) on Testing Set: {r2:.4f}")
